In [23]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import time
import psutil
from pyspark.sql import SparkSession
import numpy as np
from pyspark.sql import Row

In [24]:
# Check data hadoop
PATH_TO_CSV = "../../hdfs/data-input/drug200.parquet"
pass_linux = 'echo 123 | sudo -S'

In [25]:
!{pass_linux} docker exec namenode hdfs dfs -mkdir -p /user/data/drug
!{pass_linux} docker cp "{PATH_TO_CSV}" namenode:/tmp/drug200.parquet
!{pass_linux} docker exec namenode hdfs dfs -put /tmp/drug200.parquet /user/data/drug
!{pass_linux} docker exec namenode hdfs dfs -ls -R /user/data/drug

[sudo] password for lenovo: [sudo] password for lenovo: Successfully copied 395MB to namenode:/tmp/drug200.parquet
[sudo] password for lenovo: put: `/user/data/drug/drug200.parquet': File exists
[sudo] password for lenovo: -rw-r--r--   2 root supergroup  395495382 2026-05-06 12:15 /user/data/drug/drug200.parquet


In [26]:
memory_use = "16g"
spark = (SparkSession.builder
    .appName("Optimized_RF_50M_Rows")
    # Allocate 12GB for drivers, leaving 4GB for the OS and Docker overhead.
    .config("spark.driver.memory", memory_use)
    .config("spark.executor.memory", memory_use)
    
    # Memory Optimization: Allocate 80% of RAM to Spark,
    # Prioritizing Execution (RF computation) over Storage (cache)
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    
    # Data partitioning: 50 million rows should be divided into at least 500 partitions.
    .config("spark.sql.shuffle.partitions", "500")
    .config("spark.default.parallelism", "500")
    
    # Avoid timeouts when processing large volumes.
    .config("spark.network.timeout", "1200s")
    .config("spark.sql.broadcastTimeout", "1200s")

    .getOrCreate())

In [27]:
print(spark.version)

4.1.1


In [28]:
# Read from HDFS
ip_namenode = '172.20.0.2'
spark.sparkContext.setCheckpointDir(f"hdfs://{ip_namenode}:9000/user/checkpoints")
hdfs_path = f"hdfs://{ip_namenode}:9000/user/data/drug/drug200.parquet"
df = spark.read.parquet(hdfs_path, header=True, inferSchema=True)
# Check load data
df = df.repartition(500) 
df.show(5)
df.printSchema()

+------+---+---+------+-----------+-------+-----+
|row_id|Age|Sex|    BP|Cholesterol|Na_to_K| Drug|
+------+---+---+------+-----------+-------+-----+
|698434| 35|  F|   LOW|       HIGH| 21.095|DrugY|
|749267| 18|  M|   LOW|       HIGH|  30.23|DrugY|
|479356| 53|  F|   LOW|     NORMAL| 15.839|DrugY|
|745917| 57|  M|  HIGH|       HIGH|  9.448|DrugB|
|641846| 44|  F|NORMAL|     NORMAL| 27.084|DrugY|
+------+---+---+------+-----------+-------+-----+
only showing top 5 rows
root
 |-- row_id: long (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- BP: string (nullable = true)
 |-- Cholesterol: string (nullable = true)
 |-- Na_to_K: float (nullable = true)
 |-- Drug: string (nullable = true)



In [29]:
# df_with_id = df.withColumn("row_id", F.monotonically_increasing_id())
# df_sharded = df_with_id.repartition(8)

In [30]:
# Sử dụng RedisCluster thay vì Redis thường
# Chỉ cần liệt kê 1 vài node mồi (seed nodes), Cluster sẽ tự tìm các node còn lại
# Create redis cluster
# import subprocess
# import sys

# username = 'default'
# password = '1234'

# try:
#     from redis.cluster import RedisCluster, ClusterNode
# except ImportError:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "redis", "--break-system-packages"])
#     from redis.cluster import RedisCluster, ClusterNode

# startup_nodes = [
#     ClusterNode("172.25.0.10", 6379),
#     ClusterNode("172.25.0.11", 6379),
#     ClusterNode("172.25.0.12", 6379)
# ]

# rc = RedisCluster(
#     startup_nodes=startup_nodes,
#     username=username,
#     password=password,
#     decode_responses=True,
#     skip_full_coverage_check=True 
# )

In [31]:
def upload_to_redis_cluster(partition_index, iterator):
    from redis.cluster import RedisCluster, ClusterNode
    
    # Cấu hình cụm Redis
    startup_nodes = [ClusterNode("172.25.0.10", 6379)]
    rc = RedisCluster(
        startup_nodes=startup_nodes, 
        password='1234', 
        decode_responses=True
    )
    
    count = 0
    batch_size = 5000 
    pipe = rc.pipeline(transaction=False)
    
    for row in iterator:
        row_dict = row.asDict()
        
        # SỬ DỤNG TRỰC TIẾP row_id TỪ FILE PARQUET
        redis_key = f"drug:{row_dict['row_id']}"
        
        # Loại bỏ các giá trị None và convert sang string để lưu vào Hash
        # Lưu ý: Ta có thể bỏ row_id ra khỏi nội dung hash nếu không cần thiết
        clean_data = {k: str(v) for k, v in row_dict.items() if v is not None}
        
        pipe.hset(redis_key, mapping=clean_data)
        
        count += 1
        if count % batch_size == 0:
            pipe.execute()
            
    pipe.execute() 
    return [f"Partition {partition_index}: nạp {count} dòng"]

In [32]:
# Get data from cluster redis
def get_full_data_from_cluster(partition_index):
    try:
        from redis.cluster import RedisCluster, ClusterNode
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "redis", "--break-system-packages"])
        from redis.cluster import RedisCluster, ClusterNode

    startup_nodes = [ClusterNode("172.25.0.10", 6379)]
    rc = RedisCluster(startup_nodes=startup_nodes, password='1234', decode_responses=True)
    
    # Lấy danh sách các node Master
    primaries = rc.get_primaries()
    data = []
    
    # Mỗi partition sẽ phụ trách quét 1 node Master để tránh trùng lặp và tải nhanh nhất
    if partition_index < len(primaries):
        node = primaries[partition_index]
        # Lấy 100 dòng mẫu để kiểm tra (không nên lấy cả 50 triệu dòng chỉ để xem field)
        scanner = rc.scan_iter(match="drug:*", target_nodes=[node])
        
        for i, key in enumerate(scanner):
            if i >= 50: break # Chỉ lấy 50 dòng mỗi node để hiển thị mẫu
            
            row_dict = rc.hgetall(key)
            if row_dict:
                row_dict['_redis_key'] = key # Giữ lại key để đối chiếu
                data.append(Row(**row_dict))
                
    return data

In [33]:
start_time_import = time.time()
# Lưu ý: Không cần repartition(8) quá khắt khe vì Cluster sẽ tự điều phối slot
# Thực thi nạp dữ liệu bằng mapPartitions
results = df.rdd.mapPartitionsWithIndex(upload_to_redis_cluster).collect()

# In kết quả kiểm tra
for res in results:
    print(res)
end_time_import = time.time()
print(f'Time imported data:{end_time_import - start_time_import}')

Partition 0: nạp 100004 dòng
Partition 1: nạp 100004 dòng
Partition 2: nạp 100004 dòng
Partition 3: nạp 100004 dòng
Partition 4: nạp 100003 dòng
Partition 5: nạp 100003 dòng
Partition 6: nạp 100002 dòng
Partition 7: nạp 100002 dòng
Partition 8: nạp 100002 dòng
Partition 9: nạp 100002 dòng
Partition 10: nạp 100002 dòng
Partition 11: nạp 100002 dòng
Partition 12: nạp 100002 dòng
Partition 13: nạp 100001 dòng
Partition 14: nạp 100001 dòng
Partition 15: nạp 100001 dòng
Partition 16: nạp 100001 dòng
Partition 17: nạp 100001 dòng
Partition 18: nạp 100001 dòng
Partition 19: nạp 100001 dòng
Partition 20: nạp 100001 dòng
Partition 21: nạp 100001 dòng
Partition 22: nạp 100001 dòng
Partition 23: nạp 100001 dòng
Partition 24: nạp 100001 dòng
Partition 25: nạp 100001 dòng
Partition 26: nạp 100001 dòng
Partition 27: nạp 100001 dòng
Partition 28: nạp 100001 dòng
Partition 29: nạp 100002 dòng
Partition 30: nạp 100002 dòng
Partition 31: nạp 100002 dòng
Partition 32: nạp 100002 dòng
Partition 33: nạp 10

In [34]:
start_time_load = time.time()
# Chạy song song trên 8 partitions
sample_rdd = spark.sparkContext.parallelize(range(8), 8).flatMap(get_full_data_from_cluster)

# Chuyển thành DataFrame
df_sample = sample_rdd.toDF()

# HIỂN THỊ ĐẦY ĐỦ CỘT
# truncate=False giúp nội dung trong cột không bị cắt bớt (...)
df_sample.show(20, truncate=False)

# Xem cấu trúc Schema (các field mà Spark nhận diện được từ Redis)
df_sample.printSchema()

end_time_load = time.time()
print(f'Time loaded data:{end_time_import - start_time_import}')

+--------+---+---+------+-----------+------------------+-----+-------------+
|row_id  |Age|Sex|BP    |Cholesterol|Na_to_K           |Drug |_redis_key   |
+--------+---+---+------+-----------+------------------+-----+-------------+
|27271720|19 |M  |NORMAL|NORMAL     |15.821000099182129|DrugY|drug:27271720|
|6741023 |19 |M  |LOW   |NORMAL     |9.904999732971191 |DrugC|drug:6741023 |
|47170832|50 |F  |NORMAL|NORMAL     |13.84000015258789 |DrugX|drug:47170832|
|16929245|59 |M  |NORMAL|NORMAL     |13.690999984741211|DrugX|drug:16929245|
|39453703|32 |F  |NORMAL|NORMAL     |29.586999893188477|DrugY|drug:39453703|
|12835290|51 |M  |LOW   |HIGH       |34.11800003051758 |DrugY|drug:12835290|
|11413840|29 |F  |LOW   |NORMAL     |32.6150016784668  |DrugY|drug:11413840|
|32889648|38 |F  |LOW   |NORMAL     |31.02899932861328 |DrugY|drug:32889648|
|43976991|45 |M  |LOW   |NORMAL     |9.234000205993652 |DrugC|drug:43976991|
|15008371|35 |F  |LOW   |HIGH       |10.17199993133545 |DrugC|drug:15008371|

In [35]:
# Tạo danh sách các row_id cần lấy (từ 1 đến 10)
target_ids = range(1, 11)

# Tạo RDD từ danh sách IDs và thực hiện truy vấn trực tiếp vào Redis
def get_specific_rows(row_id):
    from redis.cluster import RedisCluster, ClusterNode
    
    startup_nodes = [ClusterNode("172.25.0.10", 6379)]
    rc = RedisCluster(startup_nodes=startup_nodes, password='1234', decode_responses=True)
    
    # Tạo key tương ứng
    redis_key = f"drug:{row_id}"
    
    # Lấy dữ liệu Hash
    data = rc.hgetall(redis_key)
    
    return data if data else None

# Thực thi song song và chuyển về DataFrame
# Chúng ta dùng 1 partition vì chỉ có 10 dòng, không cần chia nhỏ hơn
specific_rdd = spark.sparkContext.parallelize(target_ids, 1).map(get_specific_rows)

# Loại bỏ các dòng None (nếu Key không tồn tại trong Redis)
df_top_10 = specific_rdd.filter(lambda x: x is not None).toDF()

# Hiển thị kết quả
df_top_10.orderBy(F.col("row_id").cast("int")).show(truncate=False)

+---+------+-----------+-----+------------------+---+------+
|Age|BP    |Cholesterol|Drug |Na_to_K           |Sex|row_id|
+---+------+-----------+-----+------------------+---+------+
|43 |HIGH  |HIGH       |DrugA|11.77400016784668 |F  |1     |
|16 |NORMAL|NORMAL     |DrugY|20.243000030517578|F  |2     |
|49 |HIGH  |HIGH       |DrugY|23.82200050354004 |M  |3     |
|15 |NORMAL|HIGH       |DrugX|12.890000343322754|M  |4     |
|16 |NORMAL|NORMAL     |DrugX|9.567999839782715 |M  |5     |
|51 |NORMAL|HIGH       |DrugY|21.54199981689453 |F  |6     |
|15 |HIGH  |HIGH       |DrugY|26.06399917602539 |M  |7     |
|51 |HIGH  |HIGH       |DrugB|13.083000183105469|M  |8     |
|38 |HIGH  |NORMAL     |DrugY|15.420000076293945|M  |9     |
|68 |LOW   |NORMAL     |DrugY|15.642999649047852|F  |10    |
+---+------+-----------+-----+------------------+---+------+

